# Tokens

> Token grid rendering for all three selection modes (gap, word, span).

In [ ]:
#| default_exp components.tokens

In [ ]:
#| export
from typing import Any, Callable, List, Optional

from fasthtml.common import Div, Span

from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

from cjm_fasthtml_tailwind.utilities.spacing import p
from cjm_fasthtml_tailwind.utilities.sizing import w, h
from cjm_fasthtml_tailwind.utilities.typography import font_size, italic
from cjm_fasthtml_tailwind.utilities.layout import position, top, left
from cjm_fasthtml_tailwind.utilities.effects import opacity
from cjm_fasthtml_tailwind.utilities.interactivity import cursor, select
from cjm_fasthtml_tailwind.utilities.transitions_and_animation import animate
from cjm_fasthtml_tailwind.utilities.transforms import translate
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, flex_wrap, gap
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_token_selector.core.constants import (
    CARET_INDICATOR_CLS, OPACITY_50_CLS, HIGHLIGHT_CLS,
)
from cjm_fasthtml_token_selector.core.config import TokenSelectorConfig
from cjm_fasthtml_token_selector.core.models import Token, TokenRenderContext, TokenSelectorState
from cjm_fasthtml_token_selector.core.html_ids import TokenSelectorHtmlIds

## Selection Helpers

Determine token visual state based on selection mode and position.

In [ ]:
#| export
def _is_token_selected(
    index:int,              # token index
    mode:str,               # selection mode
    anchor:int,             # anchor position
    focus:int,              # focus position
) -> bool:  # whether the token is in the selection
    """Check if a token at the given index is within the current selection."""
    if mode == "gap":
        return False  # gap mode selects positions between tokens, not tokens themselves
    elif mode == "word":
        return index == anchor
    elif mode == "span":
        lo, hi = min(anchor, focus), max(anchor, focus)
        return lo <= index <= hi
    return False

In [ ]:
# Gap mode: no tokens selected
assert _is_token_selected(0, "gap", 0, 0) is False
assert _is_token_selected(5, "gap", 5, 5) is False

# Word mode: only anchor token
assert _is_token_selected(3, "word", 3, 3) is True
assert _is_token_selected(2, "word", 3, 3) is False

# Span mode: range inclusive
assert _is_token_selected(3, "span", 2, 5) is True
assert _is_token_selected(2, "span", 2, 5) is True
assert _is_token_selected(5, "span", 2, 5) is True
assert _is_token_selected(1, "span", 2, 5) is False
assert _is_token_selected(6, "span", 2, 5) is False
# Reversed anchor/focus
assert _is_token_selected(3, "span", 5, 2) is True

print("_is_token_selected tests passed!")

_is_token_selected tests passed!


In [ ]:
#| export
def _build_token_render_context(
    token:Token,                    # token being rendered
    mode:str,                       # selection mode
    anchor:int,                     # anchor position
    focus:int,                      # focus position
) -> TokenRenderContext:  # render context for styling callback
    """Build a render context for the per-token styling callback."""
    return TokenRenderContext(
        token=token,
        is_selected=_is_token_selected(token.index, mode, anchor, focus),
        is_anchor=(token.index == anchor),
        is_focus=(token.index == focus),
        selection_mode=mode,
    )

## Caret Indicator

In [ ]:
#| export
def _render_caret_indicator() -> Any:  # caret indicator Div element
    """Render the pulsing caret indicator bar."""
    return Div(cls=CARET_INDICATOR_CLS)

## Token Rendering

In [ ]:
#| export
def render_token(
    token:Token,                                      # token to render
    config:TokenSelectorConfig,                       # config for this instance
    ids:TokenSelectorHtmlIds,                         # HTML IDs
    state:Optional[TokenSelectorState] = None,        # current state for highlighting
    style_callback:Optional[Callable] = None,         # (TokenRenderContext) -> str for extra CSS
    read_only:bool = False,                           # disable interaction
) -> Any:  # Span element for this token
    """Render a single interactive word token."""
    anchor = state.anchor if state else 0
    focus = state.focus if state else 0
    mode = config.selection_mode
    is_read_only = read_only or config.read_only
    idx = token.index

    # Determine visual state
    selected = _is_token_selected(idx, mode, anchor, focus)
    show_caret = (mode == "gap" and idx == anchor and state is not None)
    dim = (mode == "gap" and idx < anchor and state is not None)

    # Build token content
    children = []
    if show_caret:
        children.append(_render_caret_indicator())
    children.append(Span(token.text))

    # Build CSS classes
    cls_parts = [
        "token", position.relative,
        p.x(1), p.y(0.5),
        border_radius.selector, cursor.pointer, select.none,
    ]
    if dim:
        cls_parts.append(OPACITY_50_CLS)
    if selected:
        cls_parts.append(HIGHLIGHT_CLS)
    if not is_read_only:
        cls_parts.append(bg_dui.base_200.hover)

    # Per-token styling callback
    extra_cls = ""
    if style_callback:
        ctx = _build_token_render_context(token, mode, anchor, focus)
        extra_cls = style_callback(ctx) or ""
    if extra_cls:
        cls_parts.append(extra_cls)

    # Click handler
    onclick = None
    if not is_read_only:
        ns = f"window.tokenSelectors['{config.prefix}']"
        if mode == "gap":
            onclick = f"{ns}.selectGap({idx})"
        elif mode == "word":
            onclick = f"{ns}.selectWord({idx})"
        elif mode == "span":
            onclick = f"{ns}.selectToken({idx})"

    return Span(
        *children,
        id=ids.token(idx),
        data_token_index=str(idx),
        cls=combine_classes(*cls_parts),
        onclick=onclick,
    )

In [ ]:
from cjm_fasthtml_token_selector.core.config import _reset_prefix_counter

_reset_prefix_counter()
cfg = TokenSelectorConfig(prefix="test")
test_ids = TokenSelectorHtmlIds(prefix="test")
t = Token("hello", 0)
s = TokenSelectorState(anchor=0, focus=0, word_count=5)

# Basic render
el = render_token(t, cfg, test_ids, s)
assert el is not None
print("render_token basic test passed!")

# Read-only: no onclick
el_ro = render_token(t, cfg, test_ids, s, read_only=True)
assert el_ro is not None
print("render_token read_only test passed!")

render_token basic test passed!
render_token read_only test passed!


## End Token

In [ ]:
#| export
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui

In [ ]:
#| export
def render_end_token(
    config:TokenSelectorConfig,                 # config for this instance
    ids:TokenSelectorHtmlIds,                   # HTML IDs
    state:Optional[TokenSelectorState] = None,  # current state
    read_only:bool = False,                     # disable interaction
) -> Any:  # end sentinel Span element
    """Render the end-of-text sentinel token."""
    word_count = state.word_count if state else 0
    anchor = state.anchor if state else 0
    mode = config.selection_mode
    is_read_only = read_only or config.read_only

    # Caret at end position (gap mode only)
    show_caret = (mode == "gap" and anchor == word_count and state is not None)

    children = []
    if show_caret:
        children.append(_render_caret_indicator())
    children.append(Span(config.end_token_text, cls=combine_classes(font_size.sm, italic)))

    cls_parts = [
        "token", position.relative,
        p.x(1), p.y(0.5),
        text_dui.base_content.opacity(30),
        cursor.pointer, select.none,
    ]
    if not is_read_only:
        cls_parts.append(text_dui.base_content.hover)

    onclick = None
    if not is_read_only and mode == "gap":
        ns = f"window.tokenSelectors['{config.prefix}']"
        onclick = f"{ns}.selectGap({word_count})"

    return Span(
        *children,
        data_token_index=str(word_count),
        cls=combine_classes(*cls_parts),
        onclick=onclick,
    )

In [ ]:
_reset_prefix_counter()
cfg = TokenSelectorConfig(prefix="test")
test_ids = TokenSelectorHtmlIds(prefix="test")
s = TokenSelectorState(anchor=5, focus=5, word_count=5)

el = render_end_token(cfg, test_ids, s)
assert el is not None
print("render_end_token test passed!")

render_end_token test passed!


## Token Grid

In [ ]:
#| export
def render_token_grid(
    tokens:List[Token],                               # token list to render
    config:TokenSelectorConfig,                       # config for this instance
    ids:TokenSelectorHtmlIds,                         # HTML IDs
    state:Optional[TokenSelectorState] = None,        # current state for highlighting
    style_callback:Optional[Callable] = None,         # (TokenRenderContext) -> str for extra CSS
    read_only:bool = False,                           # disable all interaction
) -> Any:  # Div containing the complete token grid
    """Render the full interactive token grid."""
    is_read_only = read_only or config.read_only

    # Render individual tokens
    token_elements = [
        render_token(t, config, ids, state, style_callback, is_read_only)
        for t in tokens
    ]

    # End token (gap mode only, if enabled)
    if config.show_end_token and config.selection_mode == "gap":
        token_elements.append(render_end_token(config, ids, state, is_read_only))

    word_count = len(tokens)

    return Div(
        *token_elements,
        id=ids.token_grid,
        data_word_count=str(word_count),
        cls=combine_classes(
            "token-container",
            flex_display, flex_wrap.wrap,
            gap(1), p(2),
        ),
    )

In [ ]:
from cjm_fasthtml_token_selector.helpers.tokenizer import tokenize

_reset_prefix_counter()
cfg = TokenSelectorConfig(prefix="test")
test_ids = TokenSelectorHtmlIds(prefix="test")
tokens = tokenize("the quick brown fox")
s = TokenSelectorState(anchor=2, focus=2, word_count=4)

grid = render_token_grid(tokens, cfg, test_ids, s)
assert grid is not None
print("render_token_grid test passed!")

# Read-only
grid_ro = render_token_grid(tokens, cfg, test_ids, read_only=True)
assert grid_ro is not None
print("render_token_grid read_only test passed!")

# Word mode
cfg_word = TokenSelectorConfig(prefix="test-w", selection_mode="word")
ids_word = TokenSelectorHtmlIds(prefix="test-w")
s_word = TokenSelectorState(anchor=1, focus=1, word_count=4)
grid_word = render_token_grid(tokens, cfg_word, ids_word, s_word)
assert grid_word is not None
print("render_token_grid word mode test passed!")

# Span mode
cfg_span = TokenSelectorConfig(prefix="test-s", selection_mode="span")
ids_span = TokenSelectorHtmlIds(prefix="test-s")
s_span = TokenSelectorState(anchor=1, focus=3, word_count=4)
grid_span = render_token_grid(tokens, cfg_span, ids_span, s_span)
assert grid_span is not None
print("render_token_grid span mode test passed!")

render_token_grid test passed!
render_token_grid read_only test passed!
render_token_grid word mode test passed!
render_token_grid span mode test passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()